In [1]:
import numpy as np

In [ ]:
data = np.loadtxt(
    "/Users/shahsaad.alam/personal_projects/test_claude/x_ancilla_probs.csv",
    delimiter=",",
)

In [5]:
bitstrings = data[:, 0:4].astype(int)
probs = data[:, 4]
print(bitstrings)
print(probs)

[[0 0 0 0]
 [1 0 0 0]
 [0 1 0 0]
 [1 1 0 0]
 [0 0 1 0]
 [1 0 1 0]
 [0 1 1 0]
 [1 1 1 0]
 [0 0 0 1]
 [1 0 0 1]
 [0 1 0 1]
 [1 1 0 1]
 [0 0 1 1]
 [1 0 1 1]
 [0 1 1 1]
 [1 1 1 1]]
[0.30943274 0.02492955 0.1554624  0.02492955 0.1554624  0.04652277
 0.06399205 0.04652277 0.02492955 0.00746027 0.04652277 0.00746027
 0.02492955 0.00746027 0.04652277 0.00746027]


In [6]:
N_AUX_QUBITS = 4
count_arr = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)
count_xor = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)

for p in range(len(probs)):
    bitstring = bitstrings[p]
    prob = probs[p]
    for i in range(N_AUX_QUBITS):
        for j in range(N_AUX_QUBITS):
            if bitstring[i] == 1 and bitstring[j] == 1:
                count_arr[i][j] += prob
            if (bitstring[i] ^ bitstring[j]) == 1:
                count_xor[i][j] += prob

In [ ]:
# count_arr = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)
# count_xor = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)

# for p in range(len(probs)):
#     bitstring = format(p, f"0{N_QUBITS}b")
#     for i in range(N_AUX_QUBITS):
#         for j in range(N_AUX_QUBITS):
#             if (
#                 bitstring[(N_QUBITS - N_DATA_QUBITS) - 1 - i] == "1"
#                 and bitstring[(N_QUBITS - N_DATA_QUBITS) - 1 - j] == "1"
#             ):
#                 count_arr[i, j] += probs[p]
# print(count_arr)

# for p in range(len(probs)):
#     bitstring = format(p, f"0{N_QUBITS}b")
#     for i in range(N_AUX_QUBITS):
#         for j in range(N_AUX_QUBITS):
#             if (
#                 bitstring[(N_QUBITS - N_DATA_QUBITS) - 1 - i] == "1"
#                 and bitstring[(N_QUBITS - N_DATA_QUBITS) - 1 - j] == "0"
#                 and i != j
#             ) or (
#                 bitstring[(N_QUBITS - N_DATA_QUBITS) - 1 - i] == "0"
#                 and bitstring[(N_QUBITS - N_DATA_QUBITS) - 1 - j] == "1"
#                 and i != j
#             ):
#                 count_xor[i, j] += probs[p]
# print(count_xor)

In [18]:
p_arr = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)

for i in range(N_AUX_QUBITS):
    for j in range(N_AUX_QUBITS):
        if i != j and abs(i - j) == 1:
            p_arr[i, j] = 0.5 - np.sqrt(
                0.25
                - (count_arr[i, j] - count_arr[i, i] * count_arr[j, j])
                / (1.0 - 2.0 * count_xor[i, j])
            )
            # the above and below are equivalent methods per papers, and give same answers
            # p_arr[i, j] = 0.5 - np.sqrt(
            #     0.25
            #     - (count_arr[i, j] - count_arr[i, i] * count_arr[j, j])
            #     / (
            #         1.0
            #         - 2.0 * (count_arr[i, i] + count_arr[j, j])
            #         + 4.0 * count_arr[i, j]
            #     )
            # )

for i in range(N_AUX_QUBITS):
    denom = 1.0
    for j in range(N_AUX_QUBITS):
        if i != j and abs(i - j) == 1:
            denom *= 1.0 - 2.0 * p_arr[i, j]
    p_arr[i, i] = 0.5 + (count_arr[i, i] - 0.5) / denom

In [19]:
p_arr

array([[0.0954915, 0.0954915, 0.       , 0.       ],
       [0.0954915, 0.3454915, 0.0954915, 0.       ],
       [0.       , 0.0954915, 0.3454915, 0.0954915],
       [0.       , 0.       , 0.0954915, 0.0954915]])

In [20]:
def calculate_angle(p):
    return (1.0 / np.pi) * np.arcsin(np.sqrt(p))


angle_arr = np.vectorize(calculate_angle)(p_arr)

In [21]:
from IPython.display import display, Markdown

for i in range(N_AUX_QUBITS):
    for j in range(i, N_AUX_QUBITS):
        if angle_arr[i, j] != 0.0:
            display(
                Markdown(
                    f"edge {i} $\\leftrightarrow$ {j}:\t $\\theta$=\t{angle_arr[i, j]:.6f}\n"
                )
            )

edge 0 $\leftrightarrow$ 0:	 $\theta$=	0.100000


edge 0 $\leftrightarrow$ 1:	 $\theta$=	0.100000


edge 1 $\leftrightarrow$ 1:	 $\theta$=	0.200000


edge 1 $\leftrightarrow$ 2:	 $\theta$=	0.100000


edge 2 $\leftrightarrow$ 2:	 $\theta$=	0.200000


edge 2 $\leftrightarrow$ 3:	 $\theta$=	0.100000


edge 3 $\leftrightarrow$ 3:	 $\theta$=	0.100000
